# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access dataset level metadata (attributes such as name/description are properties of the Dataset, not the dict)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print("Published: ", getattr(dataset.metadata, 'datePublished', None))
print("Identifier: ", getattr(dataset.metadata, 'identifier', None))
print("License: ", getattr(dataset.metadata, 'license', None))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List the available record sets, their @id, and fields
record_sets = dataset.metadata.find_by_type('RecordSet')
print(f"Number of record sets: {len(record_sets)}\n")
for idx, rs in enumerate(record_sets):
    print(f"Record Set #{idx+1}")
    print(f"  @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    # List fields for this record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        field_obj = dataset.metadata.find_by_id(f['@id']) if isinstance(f, dict) else dataset.metadata.find_by_id(f)
        print(f"    - @id: {field_obj['@id']}, name: {field_obj.get('name', '')}, dataType: {field_obj.get('dataType', '')}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all available RecordSet @ids
record_set_ids = [rs['@id'] for rs in record_sets]
print("Available RecordSets @id:")
for rs_id in record_set_ids:
    print(" -", rs_id)

dataframes = {}

# Load all record sets into DataFrames
for rs_id in record_set_ids:
    try:
        records_iter = dataset.records(record_set=rs_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records for {rs_id}")
        else:
            print(f"Warning: No records found for {rs_id}")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# Choose the first record set for demonstration
if record_set_ids:
    selected_record_set = record_set_ids[0]
    if selected_record_set in dataframes:
        print(f"\nColumns for {selected_record_set}:")
        print(dataframes[selected_record_set].columns.tolist())
        dataframes[selected_record_set].head()
    else:
        print("No data loaded for the selected record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Explore available numeric and grouping fields in the selected RecordSet
df = dataframes[selected_record_set]
print("Available columns:", df.columns.tolist())

# Try to find a numeric field automatically (e.g. coverage percent, MW, peptides, etc.)
import numpy as np
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
print("Numeric columns:", numeric_cols)

# Use first numeric field found, or select a typical field name
numeric_field = numeric_cols[0] if numeric_cols else None

if numeric_field:
    print(f"Selected numeric field for analysis: {numeric_field}")

    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")
    
    # Normalization
    filtered_df["%s_normalized" % numeric_field] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical column if present
    object_cols = df.select_dtypes(include='object').columns.tolist()
    group_field = None
    # Prefer a column with 'type', 'group', or 'label', or just the first object col
    for key in ['type', 'group', 'label', 'description']:
        for oc in object_cols:
            if key in oc.lower():
                group_field = oc
                break
        if group_field:
            break
    if group_field is None and object_cols:
        group_field = object_cols[0]

    if group_field:
        # Dropna because some group fields may have None
        grouped_df = filtered_df.dropna(subset=[group_field]).groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
        print(f"\nGrouped data by '{group_field}' (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    # Distribution of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # Boxplot by group_field, if group_field exists
    if group_field and df[group_field].nunique() < 30:
        plt.figure(figsize=(12,4))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We used `mlcroissant` to load the dataset *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells*.
- The dataset's Croissant schema describes several record sets containing detailed protein and experimental data with rich metadata.
- Data exploration identified main numeric fields (such as peptide or abundance values), allowed filtering and normalization, and illustrated the distribution of values and grouped statistics.
- Further analysis (e.g., downstream statistical modeling or domain-specific filtering) can be built upon these foundations using the Croissant metadata to ensure full traceability and reproducibility.

All references to data structures use their Croissant schema `@id` to ensure clarity and reproducibility in code.